### Step0: Preparation

In [ ]:
!pip install pandas
import os
pwd = os.getcwd()

### Step1: Show all databases on the Seqslab platform

In [ ]:
# import pandas as pd

# Get list of databases
databases_df = spark.sql("SHOW DATABASES").toPandas()

# Add index column (1-based like in R)
databases_df["index"] = range(1, len(databases_df) + 1)

# Reorder columns to match R output
databases_df = databases_df[["index", "namespace"]]

# Print with no row names, similar to R's print(..., row.names = FALSE)
print(databases_df.to_string(index=False, line_width=200))

# Create db_dict: index as string -> namespace
db_dict = dict(zip(databases_df["index"].astype(str), databases_df["namespace"]))

### Step2: Select the database and view  tables schema

In [ ]:
import re

# Dictionary mapping for databases
db_index = input("Enter the index of the database: ")
db = db_dict[db_index]

# Switch to the selected database
spark.sql(f"USE {db}")

# List all tables in the database
tables = spark.catalog.listTables()

# Filter tables containing "_delta_"
#matched_items = [table.name for table in tables if "_delta_" in table.name]

matched_items = [table.name for table in tables if "_delta_" in table.name]

# Collect and describe matched tables
results = {}
for tb in matched_items:
    desc = spark.sql(f"DESCRIBE {tb}")
    results[tb] = desc

# Print results
print(f"\n--- Database: {db} ---")
for name, tbl in results.items():
    print(f"\n--- Table: {name} ---")
    tbl.show(truncate=False)

### Step3: Query Data

#### Method 1 (Recommended for users familiar with Python)

In [ ]:
# Recommended for users familiar with Python
# 1️⃣ Load the table from Spark
# You can replace the table name below with your own table name from the Spark catalog.
table = spark.table("metagenomicgenefamilies_demo_delta_1115081894")

# 2️⃣ Filter, select, and persist (cache) the DataFrame
# - To change the filtering condition, modify the value in .filter(), e.g.:
#     table.species == "Escherichia coli"
# - To add more filter conditions, use logical operators like & (and) or | (or), for example:
#     table.filter((table.species == "Bacteroides vulgatus") & (table.abundanceRPKs > 10))
# - To select different columns, edit the list in .select(), e.g.:
#     .select("geneFamilyName", "abundanceRPKs", "sampleId")
# - The .persist() method caches the data in memory to speed up repeated access.
#   If you only run the query once, you can remove .persist() to save memory.
table = (
    table
    .filter(table.species == "Bacteroides vulgatus")
    .select("geneFamilyName", "abundanceRPKs", "species")
    .persist()  # cache the DataFrame for faster reuse
)

# 3️⃣ Display the result
# You can adjust the number of rows shown by adding a number inside show(), e.g. table.show(20)
table.show()

#### Method 2 (Recommended for users familiar with SQL)

In [ ]:
# Recommended for users familiar with SQL
# You can modify the SQL statement below to fit your requirements.
query="""
SELECT 
    *
FROM 
    metagenomicgenefamilies_demo_delta_1115081894 
"""
table = spark.sql(query)
table.show()

### Step4: Convert Spark table to local pandas DataFrame

In [ ]:
# To collect the dataset into local memory for further plotting or statistical analysis.
local_table = table.toPandas()
local_table